In [ ]:
# Install required packages
!pip install -q pyspellchecker stanza

# Import libraries
import requests
import stanza
from spellchecker import SpellChecker

print("Libraries installed successfully.")

In [ ]:
# Download German word list
dwds_url = "https://gist.githubusercontent.com/MarvinJWendt/2f4f4154b8ae218600eb091a5706b5f4/raw/36b70dd6be330aa61cd4d4cdfda6234dcb0b8784/wordlist-german.txt"
response = requests.get(dwds_url)

if response.status_code == 200:
    with open("word_list.txt", "w", encoding="utf-8") as file:
        file.write(response.text)
    print("Word list successfully downloaded.")
else:
    print("Failed to download word list.")

# Load word list into PySpellChecker
spell = SpellChecker(language=None)
spell.word_frequency.load_text_file("word_list.txt")
print("German word list loaded into PySpellChecker.")


In [ ]:
# OCR correction function
def correct_ocr_errors(text):
    words = text.split()
    corrected_text = " ".join([spell.correction(word) if spell.correction(word) else word for word in words])
    return corrected_text

In [ ]:
# Load sample OCR text from GitHub
github_raw_url = "https://raw.githubusercontent.com/MonikaBarget/atr-historical-research/refs/heads/main/sample_data_txt/DeutscheKolonialZeitung.txt"
response = requests.get(github_raw_url)

if response.status_code == 200:
    input_text = response.text
    corrected_text = correct_ocr_errors(input_text)
    print("Corrected OCR Text (Preview):\n", corrected_text[:500])
else:
    print("Failed to access sample text.")

In [ ]:
# Download and load Stanza German NER model (without widget)
stanza.download("de", verbose=False)  # disables widget-style progress bars
nlp = stanza.Pipeline(lang="de", processors="tokenize,mwt,ner", verbose=False)

In [ ]:
# NER extraction function
def extract_named_entities(text):
    doc = nlp(text)
    named_entities = {"Person": [], "Location": []}
    for ent in doc.ents:
        if ent.type == "PER":
            named_entities["Person"].append(ent.text)
        elif ent.type in ["LOC", "GPE"]:
            named_entities["Location"].append(ent.text)
    return named_entities

# Apply NER
entities = extract_named_entities(corrected_text)
print("Named Entities (Preview):\n", entities)
